# EU-ETS Carbon Credit Analysis

**Final Project 3, AY 2023-2024**

Authors: R.E. Coppola, C. Carzaniga, A. Torazzi

---

This notebook reproduces the full analysis pipeline originally implemented in MATLAB.
Each section corresponds to a major step in the project:

1. **Data Import and Bootstrap** -- OIS curve stripping
2. **Liquidity Analysis** -- futures volume comparison
3. **C-spread** -- convenience-yield spread (front, next, rollover)
4. **Z-spread / Z-index** -- corporate bond credit spread
5. **Time Series Alignment and Stationarity** -- KPSS / ADF-GLS tests
6. **Cointegration** -- Johansen test and Error Correction Model
7. **Variance Estimation** -- GARCH(1,1) vs EWMA
8. **Quantile Regression** -- conditional quantile analysis

In [8]:
import sys
import os
import time
import warnings
import numpy as np
from datetime import date

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(ROOT_DIR)

DATA_DIR = os.path.join(ROOT_DIR, 'data')

warnings.filterwarnings('ignore')

# Project modules
from src.utils import to_datenum, from_datenum
from src.preprocessing import DataPreprocessor
from src.bootstrap import BootstrapEngine
from src.cspread import CspreadAnalysis
from src.zspread import ZspreadAnalysis
from src.liquidity import LiquidityAnalysis
from src.timeseries import TimeSeriesAnalysis
from src.cointegration import CointegrationAnalysis
from src.variance import VarianceEstimation
from src.quantile_regression import QuantileRegressionAnalysis
from src.plotting import Plotter

# Base directory (same folder as this notebook)
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(os.path.abspath(os.path.join(os.getcwd(), '..')), 'data')

t_start = time.time()
print('Base directory:', BASE_DIR)

Base directory: c:\Users\altor\OneDrive\Desktop\pppppppppppppppppppppppppppppp\FE\FE_FinalProject\FinalProject_DEFpy\notebooks


## Configuration

In [2]:
# Analysis window (2-digit Dec contract years)
START_YEAR = 13
END_YEAR   = 20

# Rolling mechanism flag:
#   0 = classic roll on Nov 15
#   1 = open-interest crossover
#   2 = 1 month before expiry
#   3 = 1 week before expiry
FLAG_ROLLING = 0

# Variance model:
#   0 = GARCH(1,1)
#   1 = EWMA(0.95)
FLAG_VARIANCE = 0

# Stationarity tests:
#   0 = KPSS only
#   1 = KPSS + ADF-GLS
FLAG_STATIONARITY = 0

In [9]:
# Initialise shared objects
preprocessor = DataPreprocessor(DATA_DIR)
plotter = Plotter(output_dir=BASE_DIR)

---
## 1. Data Import and Bootstrap

In [10]:
# Load OIS swap rates
ois_rates, dates_matrix = preprocessor.load_ois()
print(f'OIS rates shape:   {ois_rates.shape}')
print(f'Dates matrix shape: {dates_matrix.shape}')

# Load daily futures (spot proxy)
daily_fut_dates, daily_fut_close = preprocessor.load_daily_futures()
print(f'Daily futures: {len(daily_fut_dates)} records')

# Load volume data
vol_march, vol_june, vol_sept = preprocessor.load_volumes()
dates_futures = preprocessor.load_futures_dates()
print(f'Volume (March) shape: {vol_march.shape}')

OIS rates shape:   (4744, 27)
Dates matrix shape: (4744, 28)
Daily futures: 2499 records
Volume (March) shape: (3132, 11)


In [11]:
# Bootstrap discount factors and zero rates
bootstrap = BootstrapEngine()
discounts, zero_rates = bootstrap.compute(dates_matrix, ois_rates)
print(f'Discounts shape: {discounts.shape}')

# Flat-extrapolate to 40 years
dates_40y, zero_rates_40y = BootstrapEngine.extend_to_40y(dates_matrix, zero_rates)
print(f'Extended to 40y:  {dates_40y.shape}')

Discounts shape: (4744, 28)
Extended to 40y:  (4744, 58)


---
## 2. Liquidity Analysis

In [ ]:
liquidity = LiquidityAnalysis(preprocessor)

front_march = liquidity.front_volumes(vol_march[:, 1:], 3, dates_futures) + 1
front_june  = liquidity.front_volumes(vol_june[:, 1:],  6, dates_futures) + 1
front_sept  = liquidity.front_volumes(vol_sept[:, 1:],  9, dates_futures) + 1
front_dec, next_dec, nextnext_dec = liquidity.volumes_dec()

print(f'Front March volumes:    {len(front_march)}')
print(f'Front December volumes: {len(front_dec)}')

# plotter.plot_liquidity_boxplots(front_march, front_june, front_sept, front_dec, next_dec, nextnext_dec)

Front March volumes:    2610
Front December volumes: 2532


---
## 3a. C-spread (Front and Next)

In [14]:
cspread_analysis = CspreadAnalysis(preprocessor)

C_front, dates_front, maturities = cspread_analysis.compute_front(
    zero_rates, dates_matrix, daily_fut_dates, daily_fut_close, START_YEAR, END_YEAR)
print(f'C-spread front: {len(C_front)} observations')

C_next, dates_next = cspread_analysis.compute_next(
    zero_rates, dates_matrix, daily_fut_dates, daily_fut_close, START_YEAR, END_YEAR)
print(f'C-spread next:  {len(C_next)} observations')

C-spread front: 1985 observations
C-spread next:  1730 observations


## 3b. C-spread Rollover

In [ ]:
oi_dates, oi_values = preprocessor.load_open_interest()

roll_dates = cspread_analysis.rolling_mechanism(
    START_YEAR, END_YEAR, maturities, oi_dates, oi_values, FLAG_ROLLING)
print('Roll dates:', [str(from_datenum(d)) for d in roll_dates])

C_spread, dates_Cspread_roll = cspread_analysis.compute_rollover(
    zero_rates, dates_matrix, daily_fut_dates, daily_fut_close,
    START_YEAR, END_YEAR, roll_dates)
print(f'C-spread rollover: {len(C_spread)} observations')

# plotter.plot_cspread(dates_front, C_front, dates_next, C_next, dates_Cspread_roll, C_spread)

Roll dates: ['2013-11-15', '2014-11-15', '2015-11-15', '2016-11-15', '2017-11-15', '2018-11-15', '2019-11-15', '2020-11-15']
C-spread rollover: 1981 observations


---
## 4. Z-spread / Z-index

In [ ]:
valid_bonds, names = preprocessor.load_bonds()
print('Issuers:', names)

zspread = ZspreadAnalysis(preprocessor)
z_index, dates_Z = zspread.compute_zindex(names, valid_bonds, dates_40y, zero_rates_40y)
print(f'Z-index: {len(z_index)} observations')

---
## 5. Time Series Alignment and Stationarity Tests

In [ ]:
ts = TimeSeriesAnalysis()

C_aligned, z_aligned, r3m, dates_ts = ts.align_series(
    zero_rates, dates_matrix, maturities, dates_Z,
    dates_Cspread_roll, C_spread, z_index)
print(f'Aligned time series: {len(dates_ts)} observations')

plotter.plot_time_series(dates_ts, C_aligned, z_aligned, r3m)
plotter.plot_acf_pacf(C_aligned, z_aligned, r3m)

In [ ]:
ts.stationarity_tests(C_aligned, z_aligned, r3m, FLAG_STATIONARITY)

---
## 6. Cointegration Analysis

In [ ]:
coint = CointegrationAnalysis()

Y = np.column_stack([C_aligned, z_aligned, r3m])
cointeg_vector, coint_result = coint.johansen_test(Y, alpha=0.1)

plotter.plot_cointegration(dates_ts, C_aligned, z_aligned, r3m, cointeg_vector)

---
## 7. Variance Estimation and Error Correction Model

In [ ]:
dates_extra, Ret_SPX, Ret_VIX, Ret_WTI = preprocessor.load_extra_variables()

var_est = VarianceEstimation()
var_garch, var_ewma, log_returns = var_est.estimate(daily_fut_dates, daily_fut_close, dates_ts)

plotter.plot_variance(dates_ts, var_garch, var_ewma, log_returns)

variance = var_garch if FLAG_VARIANCE == 0 else var_ewma

In [ ]:
ecm_result, x_ecm, delta_C, dates_lag, BIC, AIC, predicted = coint.error_correction_model(
    C_aligned, z_aligned, r3m, cointeg_vector,
    dates_extra, Ret_SPX, Ret_VIX, Ret_WTI,
    variance, dates_ts, FLAG_VARIANCE)

---
## 8. Quantile Regression

In [ ]:
qr = QuantileRegressionAnalysis()

x_qr = np.column_stack([np.ones(x_ecm.shape[0]), x_ecm])
quantiles = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

C_qr, P_qr, SE_qr = qr.multi_quantile_regression(x_qr, delta_C, quantiles)

In [ ]:
# Display quantile regression results
import pandas as pd

for i, q in enumerate(quantiles):
    print(f'\nQuantile {q:.1f}')
    print(f'  Coefficients: {C_qr[:, i]}')
    print(f'  P-values:     {P_qr[:, i]}')

plotter.plot_confidence_interval(x_qr, delta_C, C_qr, dates_lag, quantiles, 0)

---
## Summary

In [ ]:
elapsed = time.time() - t_start
print(f'\nCompleted in {elapsed:.1f} seconds')